# Predicting Booking Cancellations for YourCabs

**Summer Training Project**

**Author:** Khushraj Singh

**Objective:** Build a machine learning model to predict booking cancellations (due to car unavailability) using the `YourCabs.csv` dataset. This notebook contains EDA, preprocessing, modeling, evaluation and insights.


## 1. Setup & Imports

Import required libraries and set plot styling.

In [ ]:
# Basic imports and plotting settings
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as mnno

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

plt.rcParams['figure.figsize'] = (9,5)
sns.set_style('whitegrid')
print('Libraries imported')

## 2. Load dataset

Load the `YourCabs.csv` file. If your file path is different, update the path below.

In [ ]:
# Update this path if your CSV is in another location
DATA_PATH = '/mnt/data/YourCabs.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## 3. Quick Data Inspection

View basic info, missing values and a missingness matrix.

In [ ]:
# Info and missing values
df.info()
print('\nNumeric summary:')
display(df.describe(include='all').T)

# Missing values per column
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing>0]
print('\nColumns with missing values:\n', missing)

# Visual missingness map (renders in notebook)
try:
    mnno.matrix(df)
    plt.show()
except Exception as e:
    print('missingno visual failed:', e)

## 4. Exploratory Data Analysis (EDA)

Check class balance and visualize some important relationships.

In [ ]:
# Target distribution
if 'Car_Cancellation' in df.columns:
    sns.countplot(x='Car_Cancellation', data=df)
    plt.title('Distribution of Car_Cancellation (0 = no, 1 = yes)')
    plt.show()
    print(df['Car_Cancellation'].value_counts(normalize=True))
else:
    print('Car_Cancellation column not found')

In [ ]:
# Travel type vs cancellation (if present)
if 'travel_type_id' in df.columns and 'Car_Cancellation' in df.columns:
    sns.countplot(x='travel_type_id', hue='Car_Cancellation', data=df)
    plt.title('Travel Type vs Cancellation')
    plt.xlabel('travel_type_id')
    plt.show()

In [ ]:
# Booking mode vs cancellation
for col in ['online_booking', 'mobile_site_booking']:
    if col in df.columns and 'Car_Cancellation' in df.columns:
        sns.countplot(x=col, hue='Car_Cancellation', data=df)
        plt.title(f'{col} vs Car_Cancellation')
        plt.show()

## 5. Feature Engineering & Preprocessing

- Drop irrelevant IDs
- Convert date columns to datetime and extract useful features
- Create `time_diff` (hours between booking and trip start)
- Impute missing values
- One-hot encode categorical variables
- Scale numerical features (if needed)

In [ ]:
# Make a copy to preprocess
df_proc = df.copy()

# Drop columns that are identifiers or mostly null (adjust if you need them)
drop_cols = [c for c in ['id', 'user_id', 'package_id', 'to_area_id', 'to_city_id', 'to_lat', 'to_long'] if c in df_proc.columns]
df_proc.drop(columns=drop_cols, inplace=True)

# Convert datetime columns
if 'from_date' in df_proc.columns:
    df_proc['from_date'] = pd.to_datetime(df_proc['from_date'], errors='coerce')
if 'booking_created' in df_proc.columns:
    df_proc['booking_created'] = pd.to_datetime(df_proc['booking_created'], errors='coerce')

# Extract datetime features if from_date exists
if 'from_date' in df_proc.columns:
    df_proc['trip_hour'] = df_proc['from_date'].dt.hour
    df_proc['trip_weekday'] = df_proc['from_date'].dt.weekday
    df_proc['trip_month'] = df_proc['from_date'].dt.month

# time_diff in hours
if 'from_date' in df_proc.columns and 'booking_created' in df_proc.columns:
    df_proc['time_diff_hours'] = (df_proc['from_date'] - df_proc['booking_created']).dt.total_seconds() / 3600.0

# booking_nature based on time_diff_hours
def booking_nature(x):
    if pd.isna(x):
        return 'unknown'
    if x < 1:
        return 'urgent'
    if x <= 6:
        return 'same_day'
    if x <= 24:
        return 'regular'
    return 'advance'

if 'time_diff_hours' in df_proc.columns:
    df_proc['booking_nature'] = df_proc['time_diff_hours'].apply(booking_nature)

# Quick imputation for remaining missing values
impute_cols = df_proc.columns[df_proc.isnull().any()].tolist()
if impute_cols:
    imputer = SimpleImputer(strategy='most_frequent')
    df_proc[impute_cols] = imputer.fit_transform(df_proc[impute_cols])

print('After preprocessing shape:', df_proc.shape)
df_proc.head()

## 6. Prepare Data for Modeling

- Select features and target
- One-hot encode categoricals
- Train/test split with stratification

In [ ]:
# Ensure target exists
TARGET = 'Car_Cancellation'
if TARGET not in df_proc.columns:
    raise ValueError(f'Target column {TARGET} not found in dataset')

X = df_proc.drop(columns=[TARGET])
y = df_proc[TARGET].astype(int)  # ensure integer labels

# Select only reasonable columns (drop raw datetime objects if present)
X = X.drop(columns=[c for c in ['from_date','booking_created'] if c in X.columns])

# One-hot encode categorical columns
X_enc = pd.get_dummies(X, drop_first=True)

# Align train/test shape safety check
print('Feature matrix shape:', X_enc.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_enc, y, test_size=0.2, random_state=42, stratify=y)
print('Train/Test sizes:', X_train.shape, X_test.shape)

## 7. Modeling & Evaluation

Train Decision Tree, Random Forest and XGBoost. We will show metrics and confusion matrices.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    print('\nTraining', name)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    results[name] = {'model': model, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    print(f'Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
    print('\nClassification Report:\n', classification_report(y_test, preds, zero_division=0))
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

## 8. Feature Importance

Plot top features from the Random Forest model.

In [ ]:
# Feature importance from the trained Random Forest (if available)
rf = results.get('Random Forest', {}).get('model', None)
if rf is not None:
    importances = rf.feature_importances_
    feat_names = X_enc.columns
    fi = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(20)
    sns.barplot(x=fi.values, y=fi.index)
    plt.title('Top 20 Feature Importances (Random Forest)')
    plt.xlabel('Importance')
    plt.show()
else:
    print('Random Forest model not found in results.')

## 9. Conclusion & Next Steps

- Summarize results and suggest next steps such as hyperparameter tuning, handling imbalance with SMOTE, or deploying the model as an API.

**Next steps (suggested):**
- Perform hyperparameter tuning with GridSearchCV or RandomizedSearchCV.
- Address class imbalance using SMOTE or class weights.
- Add time-series features (holidays, events) and external data (traffic/weather).
- Deploy the best model using Flask/Django or as a cloud endpoint.